# AquaSense — all-in-one synthetic image study

**Research preparation only. All images, color responses, concentrations, intervals, and model scores are simulated.** There is no manufactured strip or verified ESP32-S3 dataset here.

This single notebook contains the source for both studies: the original 500-image baseline and the expanded 1,800-image scenario experiment. It covers rendering, image preprocessing/QC, RGB–HSV–LAB feature extraction, four-model tuning and training, grouped testing, calibration, charts, reporting, and one-image prediction. It does not import project `.py` files or need pre-existing datasets/models.

Download this notebook alone into an empty working folder, open it in Jupyter or Colab, and **Run All**. On a fresh folder it automatically creates `datasets/`, `models/`, `results/`, `charts/` and `scenario_study/`, then trains and evaluates everything. A full run can take considerable time and disk space. Later runs load those saved outputs unless `REBUILD_ALL = True` is set. Keep the original and expanded results separate when writing a paper.

## 1. Controls and optional dependency setup

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
ROOT = Path.cwd()
expected_outputs = [ROOT/'datasets/labels.csv', ROOT/'results/test_metrics.csv',
                    ROOT/'models/reliable/config.json', ROOT/'scenario_study/labels.csv',
                    ROOT/'scenario_study/results/metrics_by_scenario_metal.csv']
REBUILD_ALL = not all(path.exists() for path in expected_outputs)
# To force retraining later, change REBUILD_ALL to True here.
packages = {'numpy':'numpy==2.2.4', 'pandas':'pandas==2.2.3', 'scipy':'scipy==1.15.3',
            'sklearn':'scikit-learn==1.6.1', 'xgboost':'xgboost==3.4.1',
            'matplotlib':'matplotlib==3.10.3', 'joblib':'joblib==1.5.0',
            'cv2':'opencv-python==5.0.0.93', 'threadpoolctl':'threadpoolctl==3.6.0'}
missing_packages = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
print('Project folder:', ROOT)
print('Rebuild all:', REBUILD_ALL)


## 2. Image simulator, preprocessing and four regression families
The seven response functions are invented assumptions. Two captures of one profile are not two independent water samples. HSV and LAB are deterministic transforms of measured RGB values.

In [ ]:
"""Deterministic, synthetic-only seven-pad image study. No real chemical calibration."""
from __future__ import annotations
import argparse
import hashlib
import json
import platform
from pathlib import Path
import cv2
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor


SEED = 20261024
METALS = ['Pb', 'Hg', 'Cd', 'As', 'Cr', 'Cu', 'Zn']
MAXIMA = np.array([150., 30., 50., 120., 180., 1500., 5000.])
TARGETS = [f'{m}_ug_L_simulated' for m in METALS]
CHANNELS = ['R', 'G', 'B', 'H', 'S', 'V', 'L_star', 'a_star', 'b_star']
FEATURES = [f'{m}_{c}' for m in METALS for c in CHANNELS]
CARD_W, CARD_H = 560, 280
FIDUCIALS = np.array([[22, 22], [538, 22], [538, 258], [22, 258]], dtype=np.float32)
SPOTS = [(65 + 71*i, 127) for i in range(7)]
REF_CENTERS = [(175, 211), (280, 211), (385, 211)]
REF_VALUES = np.array([235., 128., 35.])
N_SESSIONS, PROFILES_PER_SESSION, CAPTURES = 50, 5, 2


def prepare_folders():
    for name in ['design', 'datasets/images', 'datasets/preprocessed', 'charts', 'models', 'results']:
        (ROOT / name).mkdir(parents=True, exist_ok=True)


def lab_from_rgb(rgb):
    value = np.asarray(rgb, dtype=float) / 255.
    linear = np.where(value <= .04045, value / 12.92, ((value+.055)/1.055)**2.4)
    xyz = linear @ np.array([[.4124564,.3575761,.1804375],[.2126729,.7151522,.0721750],[.0193339,.1191920,.9503041]]).T
    xyz /= np.array([.95047, 1., 1.08883])
    delta = 6/29
    f = np.where(xyz > delta**3, np.cbrt(xyz), xyz/(3*delta**2)+4/29)
    return np.array([116*f[1]-16, 500*(f[0]-f[1]), 200*(f[1]-f[2])])


def assay_colors(fractions, ph, temperature, lot_shift):
    # INVENTED dominant per-metal responses. These are not measured reagent colors.
    directions = np.array([[-80,-125,-30],[-20,-90,-140],[-10,-135,-85],[-115,-25,-105],[-120,-70,-15],[-10,-85,-135],[-85,-100,-20]])
    response = 1 - np.exp(-1.8 * fractions)
    cross = .045*np.roll(fractions, 1) + .025*np.roll(fractions, -1)
    rgb = np.array([238.,233.,222.]) + (response+cross)[:,None]*directions
    rgb += (ph-7)*np.array([1.3,-1.,.8]) + (temperature-25)*np.array([-.15,.1,.12])
    return np.clip(rgb+lot_shift, 40, 245)


def render_card(colors, rng):
    card = np.full((CARD_H,CARD_W,3), 242., dtype=np.float32)
    card += rng.normal(0,.9,(CARD_H,CARD_W,1)).astype(np.float32)
    card = np.clip(card,0,255).astype(np.uint8)
    for x,y in FIDUCIALS.astype(int):
        cv2.rectangle(card,(x-10,y-10),(x+10,y+10),(12,12,12),-1)
    cv2.putText(card,'AquaSense - proposed seven-spot strip',(68,57),cv2.FONT_HERSHEY_SIMPLEX,.55,(50,70,85),1,cv2.LINE_AA)
    yy,xx = np.ogrid[:CARD_H,:CARD_W]
    for i,(x,y) in enumerate(SPOTS):
        mask = (xx-x)**2+(yy-y)**2 <= 23**2
        grain = rng.normal(0,1.8,(mask.sum(),1))
        radial = (((xx-x)**2+(yy-y)**2)[mask]/23**2)[:,None]
        card[mask] = colors[i] + grain + radial*1.8
        cv2.circle(card,(x,y),24,(180,180,180),1,cv2.LINE_AA)
        cv2.putText(card,METALS[i],(x-12,174),cv2.FONT_HERSHEY_SIMPLEX,.48,(45,45,45),1,cv2.LINE_AA)
    for (x,y),value in zip(REF_CENTERS,REF_VALUES):
        cv2.rectangle(card,(x-32,y-13),(x+32,y+13),(float(value),)*3,-1)
    cv2.putText(card,'SYNTHETIC ONLY - assumed chemistry',(119,246),cv2.FONT_HERSHEY_SIMPLEX,.39,(80,80,80),1,cv2.LINE_AA)
    return np.clip(card,0,255).astype(np.uint8)


def camera_capture(card, rng, session_gain):
    angle = rng.uniform(-10,10)
    scale = rng.uniform(.88,1.)
    matrix = cv2.getRotationMatrix2D((CARD_W/2,CARD_H/2),angle,scale)
    matrix[:,2] += [320-CARD_W/2+rng.uniform(-12,12),240-CARD_H/2+rng.uniform(-12,12)]
    corners = np.array([[0,0],[CARD_W-1,0],[CARD_W-1,CARD_H-1],[0,CARD_H-1]],np.float32)
    dst = np.c_[corners,np.ones(4)] @ matrix.T + rng.uniform(-5,5,(4,2))
    homography = cv2.getPerspectiveTransform(corners,dst.astype(np.float32))
    canvas = np.empty((480,640,3),np.float32)
    canvas[:] = rng.uniform([25,35,42],[40,50,62])
    canvas += rng.normal(0,1.5,(480,640,1)).astype(np.float32)
    warped = cv2.warpPerspective(card,homography,(640,480)).astype(np.float32)
    mask = cv2.warpPerspective(np.full((CARD_H,CARD_W),255,np.uint8),homography,(640,480))
    alpha = mask[:,:,None]/255.
    canvas = canvas*(1-alpha)+warped*alpha
    yy,xx = np.mgrid[:480,:640]
    slope_x,slope_y = rng.uniform(-.08,.08,2)
    light = rng.uniform(.91,1.04)*(1+slope_x*(xx/640-.5)+slope_y*(yy/480-.5))
    gain = session_gain*rng.normal(1,.012,3)
    canvas = canvas*light[:,:,None]*gain
    gamma = rng.uniform(.96,1.04)
    canvas = 255*np.clip(canvas/255,0,1)**gamma
    noise = rng.uniform(.6,2.)
    canvas += rng.normal(0,noise,canvas.shape)
    sigma = rng.uniform(.15,.65)
    canvas = cv2.GaussianBlur(np.clip(canvas,0,255).astype(np.uint8),(3,3),sigma)
    quality = int(rng.integers(78,96))
    ok,buffer = cv2.imencode('.jpg',cv2.cvtColor(canvas,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,quality])
    assert ok
    return buffer, {'angle_deg':float(angle),'scale':float(scale),'gamma':float(gamma),'jpeg_quality':quality,'noise_sd':float(noise),'homography':homography.tolist()}


def generate_images():
    prepare_folders()
    rng = np.random.default_rng(SEED)
    rows, metadata = [], {}
    for session in range(N_SESSIONS):
        session_gain = rng.normal(1,.025,3)
        lot_shift = rng.normal(0,1.5,(7,3))
        for sample in range(PROFILES_PER_SESSION):
            sample_id = f'SIM-S{session:02d}-P{sample:02d}'
            fraction = .88*rng.beta(1.2,1.5,7)+.12*rng.beta(2,2)
            fraction[rng.random(7)<.06] = rng.uniform(0,.015)
            ph,temperature = rng.uniform(6,8),rng.uniform(20,30)
            colors = assay_colors(fraction,ph,temperature,lot_shift)
            # Same underlying strip texture/chemistry for both captures of a profile.
            card = render_card(colors,rng)
            for capture in range(CAPTURES):
                image_id = f'{sample_id}-C{capture}'
                buffer,render_info = camera_capture(card,rng,session_gain)
                path = f'datasets/images/{image_id}.jpg'
                buffer.tofile(str(ROOT/path))
                rows.append({'image_id':image_id,'sample_id':sample_id,'session_id':f'SIM-S{session:02d}',
                             'capture':capture,'image_path':path,'is_synthetic':True,
                             **dict(zip(TARGETS,fraction*MAXIMA))})
                metadata[image_id] = {'ph_simulated':float(ph),'temperature_C_simulated':float(temperature),**render_info}
    labels = pd.DataFrame(rows)
    tr,te = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(labels,groups=labels.session_id))
    labels['partition'] = 'train'
    labels.loc[te,'partition'] = 'test'
    assert len(labels)==500 and labels.sample_id.nunique()==250
    labels.to_csv(ROOT/'datasets/labels.csv',index=False,float_format='%.9f')
    (ROOT/'datasets/render_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    template = render_card(assay_colors(np.full(7,.45),7,25,np.zeros((7,3))),np.random.default_rng(SEED))
    cv2.imwrite(str(ROOT/'design/numerical_template.png'),cv2.cvtColor(template,cv2.COLOR_RGB2BGR))
    print('Generated 500 JPEGs, 250 profiles, 50 acquisition sessions.',flush=True)
    return labels


def preprocess_image(path):
    """Read pixels only. No labels, simulated pose or rendering metadata are accepted."""
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f'Unreadable image: {path}')
    gray = cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
    _,binary = cv2.threshold(gray,65,255,cv2.THRESH_BINARY_INV)
    contours,_ = cv2.findContours(binary,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if not 130 < area < 650:
            continue
        (_, _),(w,h),_ = cv2.minAreaRect(contour)
        if min(w,h)<10 or max(w,h)/min(w,h)>1.35 or area/(w*h)<.75:
            continue
        moments = cv2.moments(contour)
        candidates.append([moments['m10']/moments['m00'],moments['m01']/moments['m00']])
    points = np.array(candidates,dtype=np.float32)
    if len(points)<4:
        raise ValueError(f'Only {len(points)} fiducials found in {path.name}')
    # Known modest rotation: extreme corner centers determine the four marks.
    sums = points.sum(axis=1)
    diff = points[:,0]-points[:,1]
    indices = [np.argmin(sums),np.argmax(diff),np.argmax(sums),np.argmin(diff)]
    if len(set(indices))!=4:
        raise ValueError('Ambiguous fiducial layout')
    ordered = points[indices]
    if cv2.contourArea(ordered)<60000:
        raise ValueError('Detected quadrilateral too small')
    transform = cv2.getPerspectiveTransform(ordered,FIDUCIALS)
    rectified = cv2.cvtColor(cv2.warpPerspective(bgr,transform,(CARD_W,CARD_H)),cv2.COLOR_BGR2RGB)
    # Small fixed median filter; extraction additionally uses robust inner-region medians.
    denoised = cv2.medianBlur(rectified,3).astype(np.float32)
    refs = np.array([np.median(denoised[y-7:y+8,x-23:x+24],axis=(0,1)) for x,y in REF_CENTERS])
    span = refs[0]-refs[2]
    if np.any(span<100):
        raise ValueError('Reference range too low')
    corrected = np.clip((denoised-refs[2])*(200/span)+35,0,255).astype(np.uint8)
    check_gray = np.median(corrected[204:219,257:304],axis=(0,1))
    features = {}
    yy,xx = np.ogrid[:CARD_H,:CARD_W]
    for metal,(x,y) in zip(METALS,SPOTS):
        roi = (xx-x)**2+(yy-y)**2 <= 16**2
        rgb = np.median(corrected[roi],axis=0)
        hsv = cv2.cvtColor((rgb/255).astype(np.float32).reshape(1,1,3),cv2.COLOR_RGB2HSV)[0,0]
        hsv[1:] *= 100
        values = np.r_[rgb,hsv,lab_from_rgb(rgb)]
        features.update({f'{metal}_{c}':float(v) for c,v in zip(CHANNELS,values)})
    qc = {'fiducial_candidates':len(points),'gray_reference_abs_error':float(np.abs(check_gray-128).mean()),'minimum_reference_span':float(span.min())}
    return features,rectified,corrected,ordered,qc


def extract_dataset(labels):
    rows,qc_rows = [],[]
    for row in labels.itertuples():
        features,rectified,corrected,points,qc = preprocess_image(ROOT/row.image_path)
        rows.append({'image_id':row.image_id,'sample_id':row.sample_id,'session_id':row.session_id,'partition':row.partition,**features})
        qc_rows.append({'image_id':row.image_id,'status':'ok',**qc})
        cv2.imwrite(str(ROOT/'datasets/preprocessed'/f'{row.image_id}.png'),cv2.cvtColor(corrected,cv2.COLOR_RGB2BGR))
    frame = pd.DataFrame(rows)
    frame.to_csv(ROOT/'datasets/features.csv',index=False,float_format='%.9f')
    pd.DataFrame(qc_rows).to_csv(ROOT/'datasets/preprocessing_qc.csv',index=False)
    print(f'Preprocessed {len(frame)} images and extracted {len(FEATURES)} features each.',flush=True)
    return frame


def model_spaces():
    estimators = {
        'PLS':(PLSRegression(scale=False,max_iter=1500),{'n_components':[2,4,8]}),
        'RandomForest':(RandomForestRegressor(n_estimators=150,random_state=SEED,n_jobs=1),{'max_depth':[None,8],'min_samples_leaf':[1,3]}),
        'XGBoost':(XGBRegressor(n_estimators=200,learning_rate=.05,random_state=SEED,n_jobs=1,objective='reg:squarederror'),{'max_depth':[2,3],'reg_lambda':[1.,5.]}),
        'RBF_SVR':(SVR(kernel='rbf',epsilon=.05),{'C':[1.,10.,100.],'gamma':['scale',.02]})}
    out = {}
    for name,(estimator,grid) in estimators.items():
        pipeline = Pipeline([('scale',StandardScaler()),('model',estimator)])
        fitted = TransformedTargetRegressor(regressor=pipeline,transformer=StandardScaler())
        out[name] = (fitted,{f'regressor__model__{k}':v for k,v in grid.items()})
    return out


def train_models(labels,features):
    frame = features.merge(labels[['image_id',*TARGETS]],on='image_id',validate='one_to_one')
    tr,te = frame[frame.partition=='train'],frame[frame.partition=='test']
    assert len(tr)==400 and len(te)==100
    assert not set(tr.session_id)&set(te.session_id)
    assert not set(tr.sample_id)&set(te.sample_id)
    folds = list(GroupKFold(n_splits=5).split(tr[FEATURES],groups=tr.session_id))
    fold_rows = []
    for fold,(a,b) in enumerate(folds):
        assert not set(tr.iloc[a].session_id)&set(tr.iloc[b].session_id)
        fold_rows.extend({'image_id':image_id,'validation_fold':fold} for image_id in tr.iloc[b].image_id)
    pd.DataFrame(fold_rows).to_csv(ROOT/'results/validation_folds.csv',index=False)
    metrics,predictions,searches,baseline = [],[],[],[]
    for metal,target,maximum in zip(METALS,TARGETS,MAXIMA):
        truth = te[target].to_numpy()
        base = np.full(len(te),tr[target].mean())
        baseline.append({'metal':metal,'MAE':mean_absolute_error(truth,base),'RMSE':mean_squared_error(truth,base)**.5,'R2':r2_score(truth,base)})
        for name,(model,grid) in model_spaces().items():
            print(f'Training {metal}: {name}',flush=True)
            search = GridSearchCV(model,grid,scoring='neg_root_mean_squared_error',cv=folds,n_jobs=4,error_score='raise')
            with joblib.parallel_backend('threading'):
                search.fit(tr[FEATURES],tr[target])
            prediction = np.asarray(search.predict(te[FEATURES])).reshape(-1)
            rmse = mean_squared_error(truth,prediction)**.5
            metrics.append({'metal':metal,'model':name,'n_test_images':len(te),'n_test_profiles':te.sample_id.nunique(),
                            'MAE':mean_absolute_error(truth,prediction),'RMSE':rmse,'R2':r2_score(truth,prediction),
                            'NRMSE_percent_full_scale':100*rmse/maximum,'validation_RMSE':-search.best_score_,
                            'negative_predictions':int((prediction<0).sum()),'is_synthetic':True})
            predictions.extend({'image_id':image_id,'sample_id':sample_id,'session_id':session_id,'metal':metal,'model':name,
                                'actual_ug_L_simulated':actual,'predicted_ug_L_simulated':pred}
                               for image_id,sample_id,session_id,actual,pred in zip(te.image_id,te.sample_id,te.session_id,truth,prediction))
            searches.append({'metal':metal,'model':name,'best_parameters':json.dumps(search.best_params_,sort_keys=True),'validation_RMSE':-search.best_score_})
            cv_frame = pd.DataFrame(search.cv_results_)
            cv_frame.to_csv(ROOT/'results'/f'cv_{metal}_{name}.csv',index=False)
            joblib.dump(search.best_estimator_,ROOT/'models'/f'{metal}_{name}.joblib')
    metrics = pd.DataFrame(metrics)
    metrics.to_csv(ROOT/'results/test_metrics.csv',index=False)
    pd.DataFrame(predictions).to_csv(ROOT/'results/test_predictions.csv',index=False)
    pd.DataFrame(searches).to_csv(ROOT/'results/best_parameters.csv',index=False)
    pd.DataFrame(baseline).to_csv(ROOT/'results/training_mean_baseline.csv',index=False)
    summary = metrics.groupby('model').agg(mean_R2=('R2','mean'),mean_NRMSE_percent_full_scale=('NRMSE_percent_full_scale','mean')).sort_values('mean_NRMSE_percent_full_scale')
    summary.to_csv(ROOT/'results/model_summary.csv')
    print(summary.to_string(),flush=True)
    return metrics


def predict_image(image_path):
    features,_,_,_,qc = preprocess_image(Path(image_path))
    X = pd.DataFrame([features],columns=FEATURES)
    rows = []
    for metal in METALS:
        for family in ['PLS','RandomForest','XGBoost','RBF_SVR']:
            model = joblib.load(ROOT/'models'/f'{metal}_{family}.joblib')
            rows.append({'metal':metal,'model':family,'predicted_ug_L_simulated':float(np.asarray(model.predict(X)).reshape(-1)[0]),'training_data':'SYNTHETIC ONLY'})
    return pd.DataFrame(rows),qc


def make_charts(labels,metrics=None):
    # Illustrations are fixed examples, not selected for favorable predictions.
    selected = labels.iloc[np.linspace(0,len(labels)-1,12,dtype=int)]
    fig,axes = plt.subplots(3,4,figsize=(16,10))
    for ax,row in zip(axes.ravel(),selected.itertuples()):
        ax.imshow(cv2.cvtColor(cv2.imread(str(ROOT/row.image_path)),cv2.COLOR_BGR2RGB))
        ax.set_title(row.image_id,fontsize=9); ax.axis('off')
    fig.suptitle('500-image study: synthetic camera scenes — assumed seven-pad chemistry',fontsize=15)
    fig.tight_layout(); fig.savefig(ROOT/'charts/image_contact_sheet.png',dpi=140); plt.close(fig)
    example = labels.iloc[0]
    _,rectified,corrected,points,_ = preprocess_image(ROOT/example.image_path)
    raw = cv2.cvtColor(cv2.imread(str(ROOT/example.image_path)),cv2.COLOR_BGR2RGB)
    detected = raw.copy()
    cv2.polylines(detected,[points.astype(int)],True,(255,65,20),2)
    for p in points.astype(int): cv2.circle(detected,tuple(p),13,(255,65,20),2)
    regions = corrected.copy()
    for x,y in SPOTS: cv2.circle(regions,(x,y),16,(230,25,25),2)
    fig,axes = plt.subplots(2,2,figsize=(13,8))
    for ax,img,title in zip(axes.ravel(),[detected,rectified,corrected,regions],['1. JPEG: detect four markers','2. Perspective correction','3. Reference normalization + denoising','4. Extract 9 color features from each of 7 regions']):
        ax.imshow(img); ax.set_title(title,fontsize=12); ax.axis('off')
    fig.suptitle('Synthetic image preprocessing — no concentration labels used',fontsize=15)
    fig.tight_layout(); fig.savefig(ROOT/'charts/preprocessing_steps.png',dpi=150); plt.close(fig)
    if metrics is not None:
        fig,axes = plt.subplots(1,2,figsize=(14,5))
        for ax,metric,title in zip(axes,['NRMSE_percent_full_scale','R2'],['Test RMSE (% simulated full scale)','Test R²']):
            metrics.pivot(index='metal',columns='model',values=metric).loc[METALS].plot.bar(ax=ax)
            ax.set_title(title); ax.set_xlabel('Metal'); ax.legend(fontsize=8)
            ax.text(.5,.5,'SYNTHETIC ONLY',transform=ax.transAxes,ha='center',alpha=.16,fontsize=24,rotation=20)
        fig.suptitle('Seven-spot image simulation: held-out session evaluation')
        fig.tight_layout(); fig.savefig(ROOT/'charts/model_comparison.png',dpi=160); plt.close(fig)


def verify_study(labels,features):
    tr,te = labels[labels.partition=='train'],labels[labels.partition=='test']
    assert len(tr)==400 and len(te)==100
    assert tr.sample_id.nunique()==200 and te.sample_id.nunique()==50
    assert tr.session_id.nunique()==40 and te.session_id.nunique()==10
    assert not set(tr.session_id)&set(te.session_id)
    assert not set(tr.sample_id)&set(te.sample_id)
    assert len(list((ROOT/'datasets/images').glob('*.jpg')))==500
    assert len(list((ROOT/'datasets/preprocessed').glob('*.png')))==500
    assert features[FEATURES].shape==(500,63) and np.isfinite(features[FEATURES].to_numpy()).all()
    assert not set(FEATURES)&set(TARGETS)
    for row in labels.itertuples():
        assert cv2.imread(str(ROOT/row.image_path)).shape==(480,640,3)
    # Re-extract a held-out image from its pixels and compare with the saved feature record.
    row = labels[labels.partition=='test'].iloc[0]
    extracted,*_ = preprocess_image(ROOT/row.image_path)
    saved = features.loc[features.image_id==row.image_id,FEATURES].iloc[0].to_numpy(dtype=float)
    assert np.allclose(saved,[extracted[c] for c in FEATURES],atol=1e-7)
    metric_path = ROOT/'results/test_metrics.csv'
    if metric_path.exists():
        metrics = pd.read_csv(metric_path)
        predictions = pd.read_csv(ROOT/'results/test_predictions.csv')
        assert len(metrics)==28 and len(predictions)==2800
        for (metal,model),part in predictions.groupby(['metal','model']):
            actual,pred = part.actual_ug_L_simulated,part.predicted_ug_L_simulated
            m = metrics[(metrics.metal==metal)&(metrics.model==model)].iloc[0]
            assert np.allclose([mean_absolute_error(actual,pred),mean_squared_error(actual,pred)**.5,r2_score(actual,pred)],m[['MAE','RMSE','R2']].to_numpy(dtype=float))
        assert len(list((ROOT/'models').glob('*.joblib')))==28
        inference,_ = predict_image(ROOT/row.image_path)
        expected = predictions[predictions.image_id==row.image_id].merge(inference,on=['metal','model'])
        assert len(expected)==28 and np.allclose(expected.predicted_ug_L_simulated_x,expected.predicted_ug_L_simulated_y)
    print('Verification passed: image counts, shapes, feature extraction and available prediction metrics.',flush=True)


def write_manifest():
    from importlib.metadata import version
    files = [p for folder in ['datasets','models','results','charts','design'] for p in (ROOT/folder).rglob('*') if p.is_file() and p.name!='manifest.json']
    manifest = {'synthetic_only':True,'seed':SEED,'images':500,'independent_profiles':250,'session_groups':50,
                'train_images':400,'test_images':100,'image_size':[640,480],'features':FEATURES,
                'concentration_maxima_ug_L_simulated':dict(zip(METALS,MAXIMA)),
                'python':platform.python_version(),
                'versions':{name:version(name) for name in ['numpy','pandas','scikit-learn','xgboost','joblib','matplotlib']},
                'opencv':cv2.__version__,'concept_image_role':'visual reference only, not source pixels for the numerical renderer',
                'sha256':{p.relative_to(ROOT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in files}}
    (ROOT/'results/manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')


def run_all(train=True):
    labels = generate_images()
    features = extract_dataset(labels)
    make_charts(labels)
    verify_study(labels,features)
    metrics = train_models(labels,features) if train else None
    make_charts(labels,metrics)
    verify_study(labels,features)
    write_manifest()




## 3. Image-quality gate and synthetic-only prediction intervals
QC rejects unreadable or unsupported captures. The later prediction intervals are calibrated on independent synthetic sessions and must not be called field confidence intervals, detection limits, or safety thresholds.

In [ ]:
"""Reject images outside the supported capture conditions before returning estimates."""
import json
from pathlib import Path
import cv2
import numpy as np

DEFAULT_LIMITS = {'min_sharpness': 35., 'max_gray_error': 8., 'max_pad_spread': 25.,
                  'max_glare_fraction': .01, 'max_dark_fraction': .02, 'max_angle_deg': 20.}

class ImageRejected(ValueError):
    def __init__(self, reasons, qc=None):
        self.reasons, self.qc = list(reasons), qc or {}
        super().__init__('; '.join(self.reasons))

def check_image(path, limits=None):
    limits = {**DEFAULT_LIMITS, **(limits or {})}
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ImageRejected(['Image cannot be decoded. Upload a JPEG or PNG.'])
    if bgr.shape[:2] != (480, 640):
        raise ImageRejected(['Capture at 640 × 480 pixels; other sizes are not calibrated.'])
    try:
        features, rectified, corrected, points, qc = preprocess_image(Path(path))
    except (ValueError, cv2.error) as error:
        raise ImageRejected(['Card alignment or reference patches failed. Check orientation, lighting and all four markers.']) from error
    gray = cv2.cvtColor(rectified, cv2.COLOR_RGB2GRAY)
    qc['sharpness'] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    qc['angle_deg'] = float(abs(np.degrees(np.arctan2(*(points[1]-points[0])[::-1]))))
    qc['card_aspect'] = float(np.linalg.norm(points[1]-points[0])/np.linalg.norm(points[3]-points[0]))
    pixels = []
    spreads = []
    yy, xx = np.ogrid[:280, :560]
    for x, y in SPOTS:
        roi = (xx-x)**2+(yy-y)**2 <= 16**2
        values = rectified[roi]
        pixels.append(values)
        spreads.append(float(np.max(np.percentile(values, 95, axis=0)-np.percentile(values, 5, axis=0))))
    pixels = np.concatenate(pixels)
    qc['glare_fraction'] = float(np.mean(np.all(pixels >= 250, axis=1)))
    qc['dark_fraction'] = float(np.mean(np.all(pixels <= 15, axis=1)))
    qc['max_pad_spread'] = max(spreads)
    reasons = []
    if qc['fiducial_candidates'] != 4:
        reasons.append('Exactly four alignment markers are required; remove distracting dark shapes.')
    if not 1.7 <= qc['card_aspect'] <= 2.8 or qc['angle_deg'] > limits['max_angle_deg']:
        reasons.append('Card orientation or perspective is outside the supported capture range.')
    if qc['sharpness'] < limits['min_sharpness']:
        reasons.append('Image is too blurry. Refocus and keep the camera still.')
    if qc['gray_reference_abs_error'] > limits['max_gray_error']:
        reasons.append('Color reference check failed. Use even lighting and visible reference patches.')
    if qc['glare_fraction'] > limits['max_glare_fraction']:
        reasons.append('Glare or overexposure covers a reaction spot.')
    if qc['dark_fraction'] > limits['max_dark_fraction']:
        reasons.append('A reaction spot is obstructed or severely underexposed.')
    if qc['max_pad_spread'] > limits['max_pad_spread']:
        reasons.append('A reaction spot has excessive shadow, texture or obstruction.')
    if reasons:
        raise ImageRejected(reasons, qc)
    return features, corrected, qc

def calibrate_quality(labels):
    """Read proper-training images only; never select limits using test outcomes."""
    sharpness, spreads, gray_errors = [], [], []
    permissive = {**DEFAULT_LIMITS, 'min_sharpness': 0, 'max_pad_spread': 1000, 'max_gray_error': 1000}
    for row in labels.itertuples():
        _, _, qc = check_image(ROOT/row.image_path, permissive)
        sharpness.append(qc['sharpness']); spreads.append(qc['max_pad_spread']); gray_errors.append(qc['gray_reference_abs_error'])
    return {**DEFAULT_LIMITS, 'min_sharpness': float(max(10, np.quantile(sharpness, .01)*.45)),
            'max_pad_spread': float(max(18, np.quantile(spreads, .99)*2)),
            'max_gray_error': float(max(8, np.quantile(gray_errors, .99)*2))}


In [ ]:
"""Validation-selected models, independent calibration and synthetic-domain checks."""
import json
import hashlib
import math
from functools import lru_cache
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from threadpoolctl import threadpool_limits

RELIABLE_DIR = ROOT/'models/reliable'
RGB_FEATURES = [f'{metal}_{c}' for metal in METALS for c in ['R', 'G', 'B']]

def load_frame():
    labels = pd.read_csv(ROOT/'datasets/labels.csv')
    features = pd.read_csv(ROOT/'datasets/features.csv')
    return labels.merge(features[['image_id', *FEATURES]], on='image_id', validate='one_to_one')

def reliability_partition(frame):
    train = frame[frame.partition == 'train'].reset_index(drop=True)
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED+73).split(train, groups=train.session_id))
    proper, calibration = train.iloc[a].copy(), train.iloc[b].copy()
    test = frame[frame.partition == 'test'].copy()
    assert [part.session_id.nunique() for part in [proper, calibration, test]] == [30, 10, 10]
    assert not set(proper.session_id) & set(calibration.session_id)
    assert not set(test.session_id) & (set(proper.session_id) | set(calibration.session_id))
    return proper, calibration, test

def small_spaces():
    """Prespecified compact search, independent of any test result."""
    spaces = model_spaces()
    spaces['PLS'][1]['regressor__model__n_components'] = [4, 8]
    spaces['RandomForest'][1]['regressor__model__max_depth'] = [None]
    spaces['XGBoost'][1]['regressor__model__reg_lambda'] = [1.]
    spaces['RBF_SVR'][1]['regressor__model__gamma'] = ['scale']
    spaces['RBF_SVR'][1]['regressor__model__C'] = [10., 100.]
    return spaces

def fit_domain(proper):
    scaler = StandardScaler().fit(proper[RGB_FEATURES])
    X = scaler.transform(proper[RGB_FEATURES])
    # Same-session captures are excluded from the reference distance calibration.
    distances = []
    for session in proper.session_id.unique():
        mask = (proper.session_id == session).to_numpy()
        nn = NearestNeighbors(n_neighbors=1).fit(X[~mask])
        distances.extend(nn.kneighbors(X[mask])[0].ravel())
    threshold = float(np.quantile(distances, .99)*1.5)
    return {'scaler': scaler, 'neighbors': NearestNeighbors(n_neighbors=1).fit(X),
            'threshold': threshold, 'columns': RGB_FEATURES}

def train_reliable():
    RELIABLE_DIR.mkdir(parents=True, exist_ok=True)
    proper, calibration, test = reliability_partition(load_frame())
    parts = []
    for name, part in [('proper_train', proper), ('calibration', calibration), ('test', test)]:
        part = part[['image_id', 'sample_id', 'session_id']].copy(); part['role'] = name; parts.append(part)
    pd.concat(parts).to_csv(ROOT/'results/reliability_split.csv', index=False)
    limits = calibrate_quality(proper)
    selected, searches, calibration_rows = {}, [], []
    folds = list(GroupKFold(n_splits=3).split(proper, groups=proper.session_id))
    for metal, target in zip(METALS, TARGETS):
        candidates = []
        for family, (model, grid) in small_spaces().items():
            print(f'Reliability training: {metal} / {family}', flush=True)
            search = GridSearchCV(model, grid, cv=folds, scoring='neg_root_mean_squared_error', n_jobs=4, error_score='raise')
            with threadpool_limits(limits=1), joblib.parallel_backend('threading'):
                search.fit(proper[FEATURES], proper[target])
            candidates.append((search.best_score_, family, search.best_estimator_, search.best_params_))
            searches.append({'metal': metal, 'model': family, 'validation_RMSE': -search.best_score_, 'best_parameters': json.dumps(search.best_params_)})
        score, family, fitted, params = max(candidates, key=lambda item: item[0])
        prediction = np.asarray(fitted.predict(calibration[FEATURES])).reshape(-1)
        residuals = np.abs(calibration[target].to_numpy()-prediction)
        # Cluster-level conformal score: maximum error across all 10 images of a session.
        session_scores = pd.DataFrame({'session': calibration.session_id, 'error': residuals}).groupby('session').error.max().to_numpy()
        alpha = .10
        rank = math.ceil((len(session_scores)+1)*(1-alpha))
        radius = float(np.sort(session_scores)[rank-1])
        joblib.dump(fitted, RELIABLE_DIR/f'{metal}.joblib')
        selected[metal] = {'family': family, 'validation_RMSE': -score, 'interval_radius_ug_L_simulated': radius,
                           'nominal_coverage': .90, 'calibration_sessions': len(session_scores)}
        calibration_rows.extend({'metal': metal, 'session_id': s, 'session_max_abs_error': v} for s,v in zip(sorted(calibration.session_id.unique()), session_scores))
    pd.DataFrame(searches).to_csv(ROOT/'results/reliability_validation.csv', index=False)
    pd.DataFrame(calibration_rows).to_csv(ROOT/'results/calibration_session_scores.csv', index=False)
    joblib.dump(fit_domain(proper), RELIABLE_DIR/'domain.joblib')
    config = {'training_data': 'SYNTHETIC ONLY', 'version': 'reliable-v1', 'proper_training_sessions': 30,
              'calibration_sessions': 10, 'test_sessions': 10, 'quality_limits': limits, 'metals': selected,
              'interval_method': '90% split-conformal with maximum absolute residual per calibration session, per metal; assumes exchangeable synthetic sessions of ten images. No real-world coverage claim.',
              'domain_check': 'Training-only standardized RGB nearest-neighbor distance. Heuristic, not proof of in-domain input.'}
    config['model_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in RELIABLE_DIR.glob('*.joblib')}
    (RELIABLE_DIR/'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    print('Reliable models and independent calibration saved.', flush=True)
    return config

@lru_cache(maxsize=1)
def load_reliable():
    config = json.loads((RELIABLE_DIR/'config.json').read_text(encoding='utf-8'))
    # Reject an incomplete or changed bundle rather than mixing model versions.
    for name, digest in config['model_sha256'].items():
        if hashlib.sha256((RELIABLE_DIR/name).read_bytes()).hexdigest() != digest:
            raise RuntimeError('Model bundle checksum mismatch. Rebuild the reliability bundle.')
    return config, {metal: joblib.load(RELIABLE_DIR/f'{metal}.joblib') for metal in METALS}, joblib.load(RELIABLE_DIR/'domain.joblib')

def estimate_checked(image_path):
    config, models, domain = load_reliable()
    features, corrected, qc = check_image(image_path, config['quality_limits'])
    X = pd.DataFrame([features], columns=FEATURES)
    scaled = domain['scaler'].transform(X[domain['columns']])
    distance = float(domain['neighbors'].kneighbors(scaled)[0][0,0])
    qc['domain_distance'] = distance; qc['domain_threshold'] = domain['threshold']
    if distance > domain['threshold']:
        raise ImageRejected(['Color features are outside the supported synthetic training domain. No estimate returned.'], qc)
    rows = []
    for metal, maximum in zip(METALS, MAXIMA):
        value = float(np.asarray(models[metal].predict(X)).reshape(-1)[0])
        info = config['metals'][metal]
        radius = info['interval_radius_ug_L_simulated']
        flags = []
        if not np.isfinite(value): flags.append('nonfinite_prediction')
        elif value < 0: flags.append('below_supported_range')
        elif value > maximum: flags.append('above_supported_range')
        lower, upper = (max(0., value-radius), value+radius) if np.isfinite(value) else (None, None)
        if upper is not None and upper > maximum: flags.append('interval_exceeds_simulated_range')
        supported = not any(f in flags for f in ['nonfinite_prediction','below_supported_range','above_supported_range'])
        rows.append({'metal': metal, 'model': info['family'], 'estimate_ug_L_simulated': value if supported else None,
                     'raw_prediction': value if np.isfinite(value) else None,
                     'lower_90': lower if supported else None, 'upper_90': upper if supported else None,
                     'status': 'estimate' if supported else 'outside_supported_range', 'flags': flags})
    result = {'status': 'accepted', 'training_data': 'SYNTHETIC ONLY', 'model_version': config['version'],
              'warning': 'Demonstration estimates from simulated chemistry. Not validated for real water or safety decisions.',
              'interval_note': config['interval_method'], 'qc': qc, 'predictions': rows}
    return result, corrected



## 4. Expanded scenarios
New profiles cover standard, low-range and mixed-extreme concentrations. An entirely unseen synthetic camera/batch shift is reserved as a challenge. Deliberately damaged images audit rejection, not regression training.

In [ ]:
"""Independent synthetic scenario study; no real strip or ESP32 measurements."""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import cv2
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from threadpoolctl import threadpool_limits


SCENARIO_SEED = 20261025
OUT = ROOT / 'scenario_study'
IMAGE_DIR = OUT / 'images'
RESULTS = OUT / 'results'
MODELS = OUT / 'models'
SCENARIOS = [('standard', 50), ('low_range', 50), ('mixed_extremes', 50), ('camera_batch_shift', 30)]


def small_spaces():
    """Four prespecified compact tuning grids; no test or challenge feedback."""
    spaces = model_spaces()
    spaces['PLS'][1]['regressor__model__n_components'] = [4, 8]
    spaces['RandomForest'][1]['regressor__model__max_depth'] = [None]
    spaces['XGBoost'][1]['regressor__model__reg_lambda'] = [1.]
    spaces['RBF_SVR'][1]['regressor__model__gamma'] = ['scale']
    spaces['RBF_SVR'][1]['regressor__model__C'] = [10., 100.]
    return spaces


def _fraction(rng, scenario):
    if scenario == 'low_range':
        return rng.beta(1.1, 3.2, 7) * .22
    if scenario == 'mixed_extremes':
        values = rng.uniform(.03, .28, 7)
        high = rng.choice(7, size=int(rng.integers(2, 5)), replace=False)
        values[high] = rng.uniform(.76, 1., len(high))
        return values
    values = .88 * rng.beta(1.2, 1.5, 7) + .12 * rng.beta(2, 2, 7)
    values[rng.random(7) < .06] = rng.uniform(0, .015)
    return values


def _shift_capture(encoded, rng):
    image = cv2.imdecode(encoded, cv2.IMREAD_COLOR).astype(np.float32)
    # Deliberate camera-domain shift, not a measured ESP32-S3 image model.
    gains = rng.uniform(.91, 1.09, 3)
    image = np.clip(image * gains, 0, 255)
    gamma = rng.uniform(.88, 1.12)
    image = 255 * (image / 255) ** gamma
    image = cv2.GaussianBlur(image.astype(np.uint8), (3, 3), rng.uniform(.2, .85))
    return cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(63, 83))])[1]


def generate():
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SCENARIO_SEED)
    rows = []
    for scenario, count in SCENARIOS:
        for session in range(count):
            partition = 'challenge' if scenario == 'camera_batch_shift' else ('test' if session >= 40 else 'train')
            session_id = f'SCN-{scenario}-{session:03d}'
            session_gain = rng.normal(1, .025 if partition != 'challenge' else .055, 3)
            lot_shift = rng.normal(0, 1.5 if partition != 'challenge' else 6., (7, 3))
            for profile in range(5):
                sample_id = f'{session_id}-P{profile}'
                fractions = _fraction(rng, scenario)
                ph, temperature = rng.uniform(6, 8), rng.uniform(20, 30)
                colors = assay_colors(fractions, ph, temperature, lot_shift)
                card = render_card(colors, rng)
                for capture in range(2):
                    image_id = f'{sample_id}-C{capture}'
                    encoded, info = camera_capture(card, rng, session_gain)
                    if partition == 'challenge':
                        encoded = _shift_capture(encoded, rng)
                    relative = Path('scenario_study/images') / f'{image_id}.jpg'
                    (ROOT / relative).write_bytes(encoded.tobytes())
                    rows.append({'image_id': image_id, 'sample_id': sample_id, 'session_id': session_id,
                                 'scenario': scenario, 'partition': partition, 'capture': capture,
                                 'image_path': relative.as_posix(), 'is_synthetic': True,
                                 'ph_simulated': ph, 'temperature_C_simulated': temperature,
                                 **dict(zip(TARGETS, fractions * MAXIMA))})
        print(f'Generated {scenario}: {count * 10} images', flush=True)
    labels = pd.DataFrame(rows)
    assert len(labels) == 1800 and labels.sample_id.nunique() == 900 and labels.session_id.nunique() == 180
    assert labels.groupby('session_id').partition.nunique().eq(1).all()
    labels.to_csv(OUT / 'labels.csv', index=False, float_format='%.9f')
    return labels


def _stress_images(labels):
    """120 additional corrupted captures, not new independent samples."""
    rows = []
    for row in labels[labels.partition == 'challenge'].groupby('session_id').head(1).itertuples():
        original = cv2.imread(str(ROOT / row.image_path))
        _, _, _, markers, _ = preprocess_image(ROOT / row.image_path)
        px, py = np.rint(markers[0]).astype(int)
        missing_marker = original.copy()
        cv2.rectangle(missing_marker, (px - 32, py - 32), (px + 32, py + 32), (240, 240, 240), -1)
        conditions = {
            'heavy_blur': cv2.GaussianBlur(original, (31, 31), 8),
            'upside_down': cv2.rotate(original, cv2.ROTATE_180),
            'missing_marker': missing_marker,
            'underexposure': np.clip(original.astype(np.float32) * .22, 0, 255).astype(np.uint8),
        }
        for condition, image in conditions.items():
            image_id = f'{row.image_id}-{condition}'
            relative = Path('scenario_study/images') / f'{image_id}.jpg'
            cv2.imwrite(str(ROOT / relative), image, [cv2.IMWRITE_JPEG_QUALITY, 90])
            rows.append({'image_id': image_id, 'source_image_id': row.image_id,
                         'session_id': row.session_id, 'condition': condition,
                         'image_path': relative.as_posix(), 'is_synthetic': True})
    frame = pd.DataFrame(rows)
    assert len(frame) == 120
    frame.to_csv(OUT / 'stress_labels.csv', index=False)
    return frame


def extract(labels, stress):
    limits = json.loads((ROOT / 'models/reliable/config.json').read_text(encoding='utf-8'))['quality_limits']
    # Set using original training data only; no new test/challenge feedback.
    rows, checks = [], []
    for n, row in enumerate(labels.itertuples(), 1):
        try:
            features, _, qc = check_image(ROOT / row.image_path, limits)
            rows.append({'image_id': row.image_id, **features})
            checks.append({'image_id': row.image_id, 'scenario': row.scenario, 'partition': row.partition,
                           'accepted': True, 'reason': '', **qc})
        except ImageRejected as error:
            checks.append({'image_id': row.image_id, 'scenario': row.scenario, 'partition': row.partition,
                           'accepted': False, 'reason': '; '.join(error.reasons), **error.qc})
        if n % 300 == 0:
            print(f'Quality checked {n}/1800', flush=True)
    for row in stress.itertuples():
        try:
            _, _, qc = check_image(ROOT / row.image_path, limits)
            accepted, reason = True, ''
        except ImageRejected as error:
            accepted, reason, qc = False, '; '.join(error.reasons), error.qc
        checks.append({'image_id': row.image_id, 'scenario': row.condition, 'partition': 'stress',
                       'accepted': accepted, 'reason': reason, **qc})
    features = pd.DataFrame(rows)
    qc = pd.DataFrame(checks)
    features.to_csv(OUT / 'features.csv', index=False, float_format='%.9f')
    qc.to_csv(OUT / 'quality_checks.csv', index=False)
    return features, qc


def evaluate(labels, features, qc):
    frame = labels.merge(features, on='image_id', how='left', validate='one_to_one')
    frame = frame.merge(qc[['image_id', 'accepted']], on='image_id', validate='one_to_one')
    train = frame[(frame.partition == 'train') & frame.accepted].copy()
    assert train.session_id.nunique() >= 100
    assert not set(train.session_id) & set(frame[frame.partition != 'train'].session_id)
    assert not set(train.sample_id) & set(frame[frame.partition != 'train'].sample_id)
    folds = list(GroupKFold(n_splits=3).split(train[FEATURES], groups=train.session_id))
    for a, b in folds:
        assert not set(train.iloc[a].session_id) & set(train.iloc[b].session_id)
    MODELS.mkdir(exist_ok=True)
    metrics, predictions, validation = [], [], []
    for metal, target, maximum in zip(METALS, TARGETS, MAXIMA):
        for family, (model, grid) in small_spaces().items():
            print(f'Training {metal}: {family}', flush=True)
            search = GridSearchCV(model, grid, cv=folds, scoring='neg_root_mean_squared_error',
                                  n_jobs=4, error_score='raise')
            with threadpool_limits(limits=1), joblib.parallel_backend('threading'):
                search.fit(train[FEATURES], train[target])
            joblib.dump(search.best_estimator_, MODELS / f'{metal}_{family}.joblib')
            validation.append({'metal': metal, 'model': family, 'validation_RMSE': -search.best_score_,
                               'best_parameters': json.dumps(search.best_params_, sort_keys=True)})
            for (partition, scenario), group in frame[(frame.partition != 'train') & frame.accepted].groupby(['partition', 'scenario']):
                pred = np.asarray(search.predict(group[FEATURES])).reshape(-1)
                truth = group[target].to_numpy()
                rmse = mean_squared_error(truth, pred) ** .5
                metrics.append({'partition': partition, 'scenario': scenario, 'metal': metal, 'model': family,
                                'n_images': len(group), 'n_profiles': group.sample_id.nunique(),
                                'n_sessions': group.session_id.nunique(), 'MAE': mean_absolute_error(truth, pred),
                                'RMSE': rmse, 'R2': r2_score(truth, pred),
                                'NRMSE_percent_full_scale': 100 * rmse / maximum, 'is_synthetic': True})
                predictions.extend({'image_id': image_id, 'partition': partition, 'scenario': scenario,
                                    'session_id': session_id, 'sample_id': sample_id, 'metal': metal, 'model': family,
                                    'actual_ug_L_simulated': actual, 'predicted_ug_L_simulated': estimated}
                                   for image_id, session_id, sample_id, actual, estimated in
                                   zip(group.image_id, group.session_id, group.sample_id, truth, pred))
    pd.DataFrame(validation).to_csv(RESULTS / 'validation.csv', index=False)
    pd.DataFrame(metrics).to_csv(RESULTS / 'metrics_by_scenario_metal.csv', index=False)
    pd.DataFrame(predictions).to_csv(RESULTS / 'predictions.csv', index=False)
    qc.groupby(['partition', 'scenario']).accepted.agg(['count', 'sum', 'mean']).to_csv(RESULTS / 'acceptance_by_scenario.csv')
    config = {'seed': SCENARIO_SEED, 'source': 'SYNTHETIC ONLY', 'primary_images': 1800,
              'independent_profiles': 900, 'sessions': 180, 'stress_augmentations': 120,
              'model_selection': 'Grouped 3-fold validation on train sessions; no test/challenge tuning.',
              'challenge': 'camera_batch_shift withheld from training and validation',
              'quality_limits_source': 'original 500-image study training/calibration split',
              'model_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in MODELS.glob('*.joblib')}}
    (OUT / 'manifest.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    return pd.DataFrame(metrics)


def report(metrics):
    acceptance = pd.read_csv(RESULTS / 'acceptance_by_scenario.csv')
    mean = metrics.groupby(['partition', 'model']).NRMSE_percent_full_scale.mean().unstack(0)
    lines = ['# Expanded synthetic scenario study', '',
             '**Simulation only.** No images, colors, metal concentrations, or accuracy values here are experimentally measured.', '',
             '1,800 newly rendered primary images = 900 independent simulated concentration profiles × 2 captures, '
             'in 180 sessions. Another 120 intentionally corrupted images reuse challenge profiles and are QC stress cases, '
             'not independent samples.', '',
             'The train set contains 1,200 primary images (120 sessions) from three scenarios. '
             'The grouped same-scenario test set contains 300 images (30 sessions). '
             'The 300-image camera/batch-shift challenge (30 sessions) is absent from all tuning and training.', '',
             '## Four-model comparison', '',
             'Mean normalized RMSE is the unweighted mean of per-metal RMSE divided by each metal’s simulated full-scale range. '
             'It is not a classification accuracy.', '',
             '| Model | Same-scenario test NRMSE % | Shift challenge NRMSE % |', '|---|---:|---:|']
    for model, row in mean.sort_values('test').iterrows():
        lines.append(f"| {model} | {row['test']:.3f} | {row['challenge']:.3f} |")
    lines += ['', '## Image acceptance', '', '| Partition | Scenario | Images | Accepted | Acceptance rate |',
              '|---|---|---:|---:|---:|']
    for row in acceptance.itertuples():
        lines.append(f'| {row.partition} | {row.scenario} | {row.count} | {row.sum} | {row.mean:.1%} |')
    lines += ['', '## Interpretation', '',
              'Compare each metal’s MAE, RMSE, and R² in `results/metrics_by_scenario_metal.csv`; '
              'never treat its R² as a detection accuracy percentage. Metrics use only QC-accepted images, '
              'so read them alongside acceptance/rejection rates. The shifted-camera challenge measures transfer '
              'to one *simulated* change, not an ESP32-S3 or real-strip result. The intentionally corrupted images '
              'test whether bad captures are withheld; they do not enlarge the training dataset.', '',
              'A high score would show that the model recovered the simulator’s invented concentration-color rules. '
              'It cannot establish chemical selectivity, detection limits, interference tolerance, or field performance. '
              'Collect physical-strip images paired with laboratory reference concentrations for those claims.', '']
    (OUT / 'REPORT.md').write_text('\n'.join(lines), encoding='utf-8')


def charts():
    metrics = pd.read_csv(RESULTS / 'metrics_by_scenario_metal.csv')
    values = metrics.groupby(['scenario', 'model']).NRMSE_percent_full_scale.mean().unstack('model')
    values = values.reindex(['standard', 'low_range', 'mixed_extremes', 'camera_batch_shift'])
    fig, ax = plt.subplots(figsize=(11, 5.5))
    values.plot.bar(ax=ax, width=.8)
    ax.set(ylabel='Mean NRMSE (% of simulated full scale)', xlabel='Synthetic scenario',
           title='Four-model errors by synthetic scenario')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Model', ncol=4, loc='upper center', bbox_to_anchor=(.5, -.2))
    fig.tight_layout()
    fig.savefig(OUT / 'scenario_model_comparison.png', dpi=180)
    plt.close(fig)
    acceptance = pd.read_csv(RESULTS / 'acceptance_by_scenario.csv')
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.bar(acceptance.scenario, acceptance['mean'] * 100, color='#118287')
    ax.set(ylabel='Images accepted (%)', ylim=(0, 105), title='Image-quality gate by synthetic scenario')
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig.savefig(OUT / 'image_acceptance.png', dpi=180)
    plt.close(fig)


def refresh_stress_qc():
    """Regenerate intentional corruptions and refresh their QC summary only."""
    labels = pd.read_csv(OUT / 'labels.csv')
    stress = _stress_images(labels)
    limits = json.loads((ROOT / 'models/reliable/config.json').read_text(encoding='utf-8'))['quality_limits']
    qc = pd.read_csv(OUT / 'quality_checks.csv')
    qc = qc[qc.partition != 'stress'].copy()
    rows = []
    for row in stress.itertuples():
        try:
            _, _, details = check_image(ROOT / row.image_path, limits)
            accepted, reason = True, ''
        except ImageRejected as error:
            accepted, reason, details = False, '; '.join(error.reasons), error.qc
        rows.append({'image_id': row.image_id, 'scenario': row.condition, 'partition': 'stress',
                     'accepted': accepted, 'reason': reason, **details})
    qc = pd.concat([qc, pd.DataFrame(rows)], ignore_index=True)
    qc.to_csv(OUT / 'quality_checks.csv', index=False)
    qc.groupby(['partition', 'scenario']).accepted.agg(['count', 'sum', 'mean']).to_csv(RESULTS / 'acceptance_by_scenario.csv')
    report(pd.read_csv(RESULTS / 'metrics_by_scenario_metal.csv'))
    charts()
    return qc[qc.partition == 'stress']


def main():
    labels = generate()
    stress = _stress_images(labels)
    features, qc = extract(labels, stress)
    metrics = evaluate(labels, features, qc)
    report(metrics)
    charts()
    print('Expanded synthetic study complete.', flush=True)




## Improved models: feature selection and robustness
Version 2 selects each metal’s model and color-feature recipe using grouped training validation. It reserves 24 independent development sessions for synthetic interval calibration. A fresh 200-image audit is generated only after selection. Existing held-out results are reported as exploratory because they have already been inspected.

In [ ]:
"""Version 2: training-only feature selection, nuisance robustness and calibration."""
import hashlib
import json
import math
from functools import lru_cache
from pathlib import Path
import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits

V2_DIR = ROOT / 'improved_study'
V2_MODELS = V2_DIR / 'models'
V2_SEED = 20261029
V2_RECIPES = ['all_clean', 'own_clean', 'neighbors_clean', 'own_augmented']


def v2_data():
    labels = pd.read_csv(ROOT / 'scenario_study/labels.csv')
    features = pd.read_csv(ROOT / 'scenario_study/features.csv')
    return labels.merge(features, on='image_id', validate='one_to_one')


def v2_split(frame):
    development = frame[frame.partition == 'train'].copy()
    calibration_groups = set()
    rng = np.random.default_rng(V2_SEED)
    for _, part in development.groupby('scenario', sort=True):
        groups = np.sort(part.session_id.unique())
        calibration_groups.update(rng.choice(groups, 8, replace=False))
    calibration = development[development.session_id.isin(calibration_groups)].copy()
    proper = development[~development.session_id.isin(calibration_groups)].copy()
    heldout = frame[frame.partition != 'train'].copy()
    assert [x.session_id.nunique() for x in (proper, calibration)] == [96, 24]
    for key in ['session_id', 'sample_id']:
        sets = [set(x[key]) for x in (proper, calibration, heldout)]
        assert all(not sets[a] & sets[b] for a, b in [(0, 1), (0, 2), (1, 2)])
    return proper.reset_index(drop=True), calibration, heldout


def v2_perturb(frame, seed):
    """Assumed RGB nuisance variation; HSV/LAB stay derived from RGB."""
    out = frame[FEATURES].copy().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    for group in sorted(frame.session_id.unique()):
        indices = np.flatnonzero(frame.session_id.to_numpy() == group)
        offsets = rng.normal(0, 3., (7, 3))
        for i in indices:
            for metal, offset in zip(METALS, offsets):
                rgb = np.clip(out.loc[i, [f'{metal}_{c}' for c in ['R', 'G', 'B']]].to_numpy(dtype=float)
                              + offset + rng.normal(0, .5, 3), 0, 255)
                hsv = cv2.cvtColor((rgb / 255).astype(np.float32).reshape(1, 1, 3), cv2.COLOR_RGB2HSV)[0, 0]
                hsv[1:] *= 100
                out.loc[i, [f'{metal}_{c}' for c in CHANNELS]] = np.r_[rgb, hsv, lab_from_rgb(rgb)]
    return out


def v2_columns(metal, recipe):
    if recipe == 'all_clean':
        return FEATURES
    pads = [metal]
    if recipe == 'neighbors_clean':
        i = METALS.index(metal)
        pads = [METALS[(i - 1) % 7], metal, METALS[(i + 1) % 7]]
    return [f'{pad}_{c}' for pad in pads for c in ['R', 'G', 'B']]


def v2_candidate(metal, family, recipe, setting):
    columns = v2_columns(metal, recipe)
    model = model_spaces()[family][0]
    if family == 'PLS':
        params = {'n_components': [2, min(8, len(columns))][setting]}
    elif family == 'RandomForest':
        params = {'min_samples_leaf': [1, 3][setting], 'max_depth': None}
    elif family == 'XGBoost':
        params = {'max_depth': [2, 3][setting], 'n_estimators': 350, 'learning_rate': .035, 'reg_lambda': 5.}
    else:
        params = {'C': [10., 100.][setting], 'gamma': 'scale', 'epsilon': .01}
    model.set_params(**{f'regressor__model__{k}': v for k, v in params.items()})
    return Pipeline([('columns', ColumnTransformer([('selected', 'passthrough', columns)])), ('regression', model)])


def train_improved():
    V2_MODELS.mkdir(parents=True, exist_ok=True)
    proper, calibration, _ = v2_split(v2_data())
    protocol = {'version': 'expanded-v2', 'seed': V2_SEED, 'recipes': V2_RECIPES,
                'proper_sessions': 96, 'calibration_sessions': 24,
                'selection': '3 grouped folds; equal-weight clean and assumed RGB-noise validation MSE; no test/challenge tuning',
                'perturbation': 'Per-session, per-pad RGB Gaussian sd=3; per-image sd=0.5; HSV/LAB recomputed; applied within training folds only',
                'old_holdout_status': 'Previously inspected; comparisons are exploratory. New audit generated only after selection.',
                'audit_seed': V2_SEED + 10000, 'audit_sessions': 20}
    (V2_DIR / 'protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
    assignment = pd.concat([proper.assign(role='proper_train'), calibration.assign(role='calibration')])
    assignment[['image_id', 'sample_id', 'session_id', 'scenario', 'role']].to_csv(V2_DIR / 'split.csv', index=False)
    folds = list(GroupKFold(3).split(proper, groups=proper.session_id))
    fold_data = []
    for fold, (a, b) in enumerate(folds):
        train, valid = proper.iloc[a], proper.iloc[b]
        assert not set(train.session_id) & set(valid.session_id)
        fold_data.append((train, valid, v2_perturb(train, V2_SEED + fold), v2_perturb(valid, V2_SEED + 100 + fold)))
    pd.concat([proper.iloc[b][['image_id', 'session_id']].assign(validation_fold=i) for i, (_, b) in enumerate(folds)]).to_csv(V2_DIR / 'validation_folds.csv', index=False)
    augmented_proper = v2_perturb(proper, V2_SEED + 500)
    validation_rows, selected, calibration_rows = [], {}, []
    for metal, target in zip(METALS, TARGETS):
        family_winners = []
        for family in ['PLS', 'RandomForest', 'XGBoost', 'RBF_SVR']:
            print(f'V2 validation: {metal} / {family}', flush=True)
            trials = []
            for recipe in V2_RECIPES:
                for setting in range(2):
                    squared_clean, squared_noise = [], []
                    for train, valid, noisy_train, noisy_valid in fold_data:
                        X, y = train[FEATURES], train[target]
                        if recipe.endswith('augmented'):
                            X = pd.concat([X, noisy_train], ignore_index=True)
                            y = np.tile(y, 2)
                        fitted = v2_candidate(metal, family, recipe, setting)
                        with threadpool_limits(limits=1):
                            fitted.fit(X, y)
                            clean_pred = np.asarray(fitted.predict(valid[FEATURES])).reshape(-1)
                            noisy_pred = np.asarray(fitted.predict(noisy_valid)).reshape(-1)
                        squared_clean.extend((clean_pred - valid[target].to_numpy()) ** 2)
                        squared_noise.extend((noisy_pred - valid[target].to_numpy()) ** 2)
                    clean = float(np.sqrt(np.mean(squared_clean)))
                    noise = float(np.sqrt(np.mean(squared_noise)))
                    score = float(np.sqrt((clean ** 2 + noise ** 2) / 2))
                    validation_rows.append({'metal': metal, 'model': family, 'recipe': recipe, 'setting': setting,
                                            'clean_RMSE': clean, 'perturbed_RMSE': noise, 'selection_RMSE': score})
                    trials.append((score, recipe, setting))
            score, recipe, setting = min(trials)
            fitted = v2_candidate(metal, family, recipe, setting)
            X, y = proper[FEATURES], proper[target]
            if recipe.endswith('augmented'):
                X, y = pd.concat([X, augmented_proper], ignore_index=True), np.tile(y, 2)
            with threadpool_limits(limits=1):
                fitted.fit(X, y)
            joblib.dump(fitted, V2_MODELS / f'{metal}_{family}.joblib')
            family_winners.append((score, family, recipe, setting, fitted))
        score, family, recipe, setting, fitted = min(family_winners, key=lambda item: item[0])
        residual = abs(np.asarray(fitted.predict(calibration[FEATURES])).reshape(-1) - calibration[target].to_numpy())
        session_scores = pd.DataFrame({'session_id': calibration.session_id.to_numpy(), 'error': residual}).groupby('session_id').error.max()
        rank = math.ceil((len(session_scores) + 1) * .9)
        radius = float(np.sort(session_scores.to_numpy())[rank - 1])
        calibration_rows.extend({'metal': metal, 'session_id': group, 'max_abs_error': error} for group, error in session_scores.items())
        selected[metal] = {'family': family, 'recipe': recipe, 'setting': setting, 'validation_RMSE': score,
                           'interval_radius_ug_L_simulated': radius, 'nominal_coverage': .9}
        joblib.dump(fitted, V2_MODELS / f'{metal}.joblib')
    pd.DataFrame(validation_rows).to_csv(V2_DIR / 'validation.csv', index=False)
    pd.DataFrame(calibration_rows).to_csv(V2_DIR / 'calibration_scores.csv', index=False)
    joblib.dump(fit_domain(proper), V2_MODELS / 'domain.joblib')
    config = {'version': 'expanded-v2', 'training_data': 'SYNTHETIC ONLY', 'metals': selected,
              'quality_limits': json.loads((ROOT / 'models/reliable/config.json').read_text())['quality_limits'],
              'proper_training_sessions': 96, 'calibration_sessions': 24,
              'interval_method': '90% per-metal session-max split-conformal on 24 clean synthetic sessions; no shifted-domain or real-water coverage guarantee.',
              'model_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in V2_MODELS.glob('*.joblib')}}
    (V2_MODELS / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    load_improved.cache_clear()
    return config


@lru_cache(maxsize=1)
def load_improved():
    config = json.loads((V2_MODELS / 'config.json').read_text(encoding='utf-8'))
    for name, digest in config['model_sha256'].items():
        if hashlib.sha256((V2_MODELS / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError('V2 model checksum mismatch. Rebuild the improved bundle.')
    return config, {metal: joblib.load(V2_MODELS / f'{metal}.joblib') for metal in METALS}, joblib.load(V2_MODELS / 'domain.joblib')


def estimate_improved(image_path):
    config, models, domain = load_improved()
    features, corrected, qc = check_image(image_path, config['quality_limits'])
    X = pd.DataFrame([features], columns=FEATURES)
    distance = float(domain['neighbors'].kneighbors(domain['scaler'].transform(X[domain['columns']]))[0][0, 0])
    qc.update(domain_distance=distance, domain_threshold=domain['threshold'])
    if distance > domain['threshold']:
        raise ImageRejected(['Color features are outside the supported synthetic training domain.'], qc)
    rows = []
    for metal, maximum in zip(METALS, MAXIMA):
        value = float(np.asarray(models[metal].predict(X)).reshape(-1)[0])
        info = config['metals'][metal]
        radius = info['interval_radius_ug_L_simulated']
        supported = bool(np.isfinite(value) and 0 <= value <= maximum)
        flags = [] if supported else ['outside_supported_range']
        if supported and value + radius > maximum:
            flags.append('interval_exceeds_simulated_range')
        rows.append({'metal': metal, 'model': info['family'], 'feature_recipe': info['recipe'],
                     'estimate_ug_L_simulated': value if supported else None,
                     'raw_prediction': value if np.isfinite(value) else None,
                     'lower_90': max(0., value - radius) if supported else None,
                     'upper_90': value + radius if supported else None,
                     'status': 'estimate' if supported else 'outside_supported_range', 'flags': flags})
    return {'status': 'accepted', 'training_data': 'SYNTHETIC ONLY', 'model_version': config['version'],
            'warning': 'Simulated chemistry only; physical-strip validation is pending.',
            'interval_note': config['interval_method'], 'qc': qc, 'predictions': rows}, corrected


def generate_v2_audit():
    """New synthetic sessions generated only after the selection bundle exists."""
    config, _, _ = load_improved()
    rng = np.random.default_rng(V2_SEED + 10000)
    folder = V2_DIR / 'audit_images'
    folder.mkdir(exist_ok=True)
    rows, checks = [], []
    for session in range(20):
        shifted = session >= 10
        scenario = 'fresh_shift' if shifted else 'fresh_nominal'
        lot = rng.normal(0, 6. if shifted else 1.5, (7, 3))
        gain = rng.normal(1, .055 if shifted else .025, 3)
        for profile in range(5):
            fractions = rng.uniform(.01, .99, 7)
            card = render_card(assay_colors(fractions, rng.uniform(6, 8), rng.uniform(20, 30), lot), rng)
            for capture in range(2):
                image_id = f'AUDIT-S{session:02d}-P{profile}-C{capture}'
                encoded, _ = camera_capture(card, rng, gain)
                if shifted:
                    image = cv2.imdecode(encoded, cv2.IMREAD_COLOR).astype(np.float32)
                    image = np.clip(image * rng.uniform(.91, 1.09, 3), 0, 255)
                    image = (255 * (image / 255) ** rng.uniform(.88, 1.12)).astype(np.uint8)
                    encoded = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 72])[1]
                path = folder / f'{image_id}.jpg'
                path.write_bytes(encoded.tobytes())
                item = {'image_id': image_id, 'session_id': f'AUDIT-S{session:02d}', 'sample_id': f'AUDIT-S{session:02d}-P{profile}',
                        'scenario': scenario, 'partition': 'fresh_audit', 'image_path': path.relative_to(ROOT).as_posix(),
                        'is_synthetic': True, **dict(zip(TARGETS, fractions * MAXIMA))}
                try:
                    features, _, _ = check_image(path, config['quality_limits'])
                    item.update(features)
                    checks.append({'image_id': image_id, 'scenario': scenario, 'accepted': True, 'reason': ''})
                except ImageRejected as error:
                    checks.append({'image_id': image_id, 'scenario': scenario, 'accepted': False, 'reason': '; '.join(error.reasons)})
                rows.append(item)
    audit = pd.DataFrame(rows)
    audit.to_csv(V2_DIR / 'fresh_audit.csv', index=False)
    pd.DataFrame(checks).to_csv(V2_DIR / 'fresh_audit_qc.csv', index=False)
    return audit.dropna(subset=FEATURES)


def evaluate_improved(audit):
    config, models, domain = load_improved()
    _, _, heldout = v2_split(v2_data())
    evaluation = pd.concat([heldout, audit], ignore_index=True)
    rows, predictions, all_predictions = [], [], []
    for metal, target, maximum in zip(METALS, TARGETS, MAXIMA):
        candidates = {'selected_v2': models[metal]}
        for family in ['PLS', 'RandomForest', 'XGBoost', 'RBF_SVR']:
            candidates[f'v1_{family}'] = joblib.load(ROOT / 'scenario_study/models' / f'{metal}_{family}.joblib')
            candidates[f'v2_{family}'] = joblib.load(V2_MODELS / f'{metal}_{family}.joblib')
        for (partition, scenario), part in evaluation.groupby(['partition', 'scenario']):
            distance = domain['neighbors'].kneighbors(domain['scaler'].transform(part[domain['columns']]))[0][:, 0]
            for name, model in candidates.items():
                pred = np.asarray(model.predict(part[FEATURES])).reshape(-1)
                truth = part[target].to_numpy()
                rmse = mean_squared_error(truth, pred) ** .5
                rows.append({'partition': partition, 'scenario': scenario, 'metal': metal, 'model': name,
                             'n_images': len(part), 'MAE': mean_absolute_error(truth, pred), 'RMSE': rmse,
                             'R2': r2_score(truth, pred), 'NRMSE_percent_full_scale': 100 * rmse / maximum})
                all_predictions.extend({'image_id': image_id, 'session_id': session_id, 'scenario': scenario,
                                        'partition': partition, 'metal': metal, 'model': name, 'actual': a, 'predicted': p}
                                       for image_id, session_id, a, p in zip(part.image_id, part.session_id, truth, pred))
                if name == 'selected_v2':
                    radius = config['metals'][metal]['interval_radius_ug_L_simulated']
                    predictions.extend({'image_id': image_id, 'session_id': session_id, 'scenario': scenario,
                                        'partition': partition, 'metal': metal, 'actual': a, 'predicted': p,
                                        'interval_covered': bool(abs(a - p) <= radius), 'domain_supported': bool(d <= domain['threshold']),
                                        'range_supported': bool(0 <= p <= maximum)}
                                       for image_id, session_id, a, p, d in zip(part.image_id, part.session_id, truth, pred, distance))
    metrics = pd.DataFrame(rows)
    metrics.to_csv(V2_DIR / 'comparison_metrics.csv', index=False)
    pd.DataFrame(all_predictions).to_csv(V2_DIR / 'all_predictions.csv', index=False)
    pred = pd.DataFrame(predictions)
    pred.to_csv(V2_DIR / 'selected_predictions.csv', index=False)
    coverage = pred.groupby(['scenario', 'metal']).agg(image_coverage=('interval_covered', 'mean'), domain_acceptance=('domain_supported', 'mean'), range_acceptance=('range_supported', 'mean'))
    sessions = pred.groupby(['scenario', 'metal', 'session_id']).interval_covered.all().groupby(['scenario', 'metal']).mean()
    coverage['session_coverage'] = sessions
    coverage.to_csv(V2_DIR / 'coverage_and_acceptance.csv')
    v2_bootstrap()
    v2_report(metrics, config)
    return metrics


def v2_bootstrap():
    """Paired session bootstrap; preserve dependence between captures and metals."""
    predictions = pd.read_csv(V2_DIR / 'all_predictions.csv')
    predictions = predictions[predictions.partition == 'fresh_audit'].copy()
    scale = predictions.metal.map(dict(zip(METALS, MAXIMA)))
    predictions['squared_normalized_error'] = ((predictions.predicted - predictions.actual) / scale) ** 2
    rng = np.random.default_rng(V2_SEED + 900)
    rows = []
    for scenario, part in predictions.groupby('scenario'):
        old = part[part.model == 'v1_XGBoost'].pivot_table(index='session_id', columns='metal', values='squared_normalized_error', aggfunc='mean').reindex(columns=METALS)
        new = part[part.model == 'selected_v2'].pivot_table(index='session_id', columns='metal', values='squared_normalized_error', aggfunc='mean').reindex(index=old.index, columns=METALS)
        indices = rng.integers(0, len(old), size=(2000, len(old)))
        changes = 100 * (np.sqrt(new.to_numpy()[indices].mean(axis=1)).mean(axis=1)
                         - np.sqrt(old.to_numpy()[indices].mean(axis=1)).mean(axis=1))
        low, high = np.percentile(changes, [2.5, 97.5])
        delta = 100 * (np.sqrt(new.mean()).mean() - np.sqrt(old.mean()).mean())
        rows.append({'scenario': scenario, 'sessions': len(old), 'paired_resamples': 2000,
                     'NRMSE_change_percentage_points': delta, 'lower_95': low, 'upper_95': high})
    pd.DataFrame(rows).to_csv(V2_DIR / 'fresh_audit_bootstrap.csv', index=False)


def v2_report(metrics, config):
    from matplotlib import pyplot as plt
    summary = metrics.groupby(['scenario', 'model']).NRMSE_percent_full_scale.mean().unstack()
    summary.to_csv(V2_DIR / 'summary.csv')
    lines = ['# Improved synthetic models (v2)', '',
             'All results concern simulated chemistry. Earlier test/challenge results informed the decision to investigate gaps; '
             'their reuse is exploratory. A fresh 200-image audit is generated after model selection, using new sessions and seed. '
             'It remains a test of the same assumed chemistry. Fresh audit concentrations are independent uniform fractions '
             'of each simulated range. "Fresh nominal" describes nominal imaging conditions, not identical concentration distributions.', '',
             'V2 reserves 24 of the 120 original development sessions for interval calibration; fits use the other 96. '
             'V1 used all 120 for fitting. Thus these versions differ in both training method and calibration allocation.', '',
             'Selection compares all-pad features, own-pad RGB, neighboring-pad RGB, and own-pad RGB with assumed nuisance augmentation. '
             'Three grouped training folds score clean and perturbed validation images equally. Scaling is fitted within each fold. '
             'Calibration and evaluation labels never select recipes or parameters.', '',
             'Neighbor features use the simulator’s cyclic cross-response mapping (including the first/last pad pair). '
             'This mapping is an assumed mathematical relationship, not measured reagent interference.', '',
             '| Metal | Selected family | Feature recipe |', '|---|---|---|']
    for metal, info in config['metals'].items():
        lines.append(f"| {metal} | {info['family']} | {info['recipe']} |")
    lines += ['', '| Scenario | Previous XGBoost NRMSE % | Selected v2 NRMSE % |', '|---|---:|---:|']
    for scenario, row in summary.iterrows():
        lines.append(f"| {scenario} | {row['v1_XGBoost']:.3f} | {row['selected_v2']:.3f} |")
    lines += ['', '| Family | V2 fresh nominal NRMSE % | V2 fresh shift NRMSE % |', '|---|---:|---:|']
    for family in ['PLS', 'RandomForest', 'XGBoost', 'RBF_SVR']:
        lines.append(f"| {family} | {summary.loc['fresh_nominal', 'v2_' + family]:.3f} | {summary.loc['fresh_shift', 'v2_' + family]:.3f} |")
    intervals = pd.read_csv(V2_DIR / 'coverage_and_acceptance.csv').groupby('scenario')[['image_coverage', 'session_coverage', 'domain_acceptance']].mean()
    lines += ['', '| Scenario | Mean image interval coverage | Mean session coverage | Domain gate acceptance |', '|---|---:|---:|---:|']
    for scenario, row in intervals.iterrows():
        lines.append(f"| {scenario} | {row.image_coverage:.1%} | {row.session_coverage:.1%} | {row.domain_acceptance:.1%} |")
    lines += ['', 'The 90% target is per metal over a new exchangeable session. Coverage under shifted imaging is substantially lower. '
              'The nearest-neighbor domain gate accepts most shifted captures, so passing that check does not demonstrate reliable uncertainty.', '',
              '| Fresh audit | NRMSE change, v2 minus v1 (percentage points) | Approximate paired 95% interval |', '|---|---:|---|']
    for row in pd.read_csv(V2_DIR / 'fresh_audit_bootstrap.csv').itertuples():
        lines.append(f'| {row.scenario} | {row.NRMSE_change_percentage_points:.3f} | [{row.lower_95:.3f}, {row.upper_95:.3f}] |')
    lines += ['', 'The percentile intervals resample whole sessions jointly across all metals (2,000 resamples). '
              'There are only ten sessions per condition. An interval spanning zero does not establish a reliable improvement.']
    lines += ['', 'NRMSE uses simulated full-scale concentrations; it is not percent-correct accuracy. '
              'Tables include all QC-accepted images before domain/range withholding. '
              'Read fresh_audit_qc.csv and coverage_and_acceptance.csv alongside errors. '
              'Intervals are calibrated on clean synthetic sessions only; shifted-domain coverage can fail. '
              'Each fresh audit condition has only ten sessions, so differences are descriptive.', '',
              'Remaining gaps: experimentally measured reagent response, real camera captures, reference-lab labels, '
              'and physical board configuration. Arbitrary pad-specific color offsets can be indistinguishable from concentration changes; '
              'more training cannot eliminate that ambiguity.', '']
    (V2_DIR / 'REPORT.md').write_text('\n'.join(lines), encoding='utf-8')
    fig, ax = plt.subplots(figsize=(11, 5))
    summary[['v1_XGBoost', 'selected_v2']].plot.bar(ax=ax)
    ax.set(ylabel='Mean RMSE / simulated full scale (%)', xlabel='', title='Synthetic model comparison: lower error is better')
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig.savefig(V2_DIR / 'comparison.png', dpi=160)
    plt.close(fig)


def run_improvement():
    train_improved()
    audit = generate_v2_audit()
    evaluate_improved(audit)
    print('Improved model study finished.', flush=True)




## 5. Plain-language interpretation

In [ ]:
"""Plain-language interpretation of synthetic AquaSense regression outputs."""

from __future__ import annotations

import pandas as pd



def interpret_prediction(result: dict) -> dict:
    """Describe an estimate without implying chemical or regulatory validity."""
    if result.get('status') != 'accepted':
        return {
            'decision': 'No result',
            'summary': 'The image did not pass the required checks; no concentrations were estimated.',
            'next_step': 'Retake the complete card under even light, with sharp visible markers and reference patches.',
            'limitations': 'The model is trained only on synthetic images and cannot validate a water sample.',
        }
    rows = result.get('predictions', [])
    withheld = [r['metal'] for r in rows if r.get('status') != 'estimate']
    range_flags = [r['metal'] for r in rows if 'interval_exceeds_simulated_range' in r.get('flags', [])]
    if withheld:
        decision = 'Partial result'
        summary = f"Estimates are withheld for {', '.join(withheld)} because their raw predictions fall outside the simulated training range."
        next_step = 'Review image quality and repeat the capture; do not replace withheld values with zero or an endpoint.'
    else:
        decision = 'Synthetic estimates available'
        summary = 'Seven simulated concentrations were estimated from the corrected strip image.'
        next_step = 'Use these values only to exercise the processing workflow. Obtain paired lab measurements before interpreting real water.'
    if range_flags:
        summary += f" Intervals for {', '.join(range_flags)} extend beyond the simulated concentration range."
    return {
        'decision': decision,
        'summary': summary,
        'next_step': next_step,
        'how_to_read': 'Each estimate is a continuous regression output in simulated µg/L. The nominal 90% interval is calibrated on clean held-out synthetic sessions. Coverage can be substantially lower under changed lighting or strip batches even when image checks pass; this is not chemical certainty, a detection limit, or a safety threshold.',
        'limitations': 'No real strip chemistry, ESP32 capture data, reference-laboratory values, or regulatory threshold has been validated.',
    }


def build_research_interpretation() -> str:
    """Rebuild a paper-ready, explicitly simulation-limited interpretation from saved CSVs."""
    metrics = pd.read_csv(ROOT / 'results/test_metrics.csv')
    summary = metrics.groupby('model', sort=False)[['R2', 'NRMSE_percent_full_scale']].mean().sort_values('NRMSE_percent_full_scale')
    lines = [
        '# Interpretation of the synthetic-image experiment', '',
        '**Scope:** 500 rendered images represent 250 simulated concentration profiles (two captures each) in 50 simulated sessions. '
        'The common held-out test set has 100 images from 50 profiles in 10 sessions. These are not physical water samples.', '',
        '## Reading the metrics', '',
        '- MAE and RMSE are in simulated µg/L for each metal. RMSE penalizes larger mistakes more strongly.',
        '- R² describes fit to held-out simulated targets; it is not classification accuracy or a percentage of correctly detected metals.',
        '- Normalized RMSE divides RMSE by that metal’s **simulated full-scale concentration**. Its mean permits an across-metal comparison; raw MAE/RMSE should not be averaged across different concentration ranges.',
        '- Predictive intervals are separate from model-error metrics. An interval is not a concentration detection limit or a probability that a specific value is chemically true.', '',
        '## Four-model held-out comparison', '',
        '| Model | Mean R² | Mean normalized RMSE (% simulated full scale) |',
        '|---|---:|---:|',
    ]
    for model, row in summary.iterrows():
        lines.append(f"| {model} | {row.R2:.3f} | {row.NRMSE_percent_full_scale:.3f} |")
    winners = metrics.loc[metrics.groupby('metal').RMSE.idxmin(), ['metal', 'model']].set_index('metal').loc[METALS]
    lines += ['', 'XGBoost has the lowest mean normalized RMSE in this one held-out synthetic test. '
              'That is a descriptive ranking, not proof that it will perform best on another simulator or a manufactured strip.', '',
              '## Per-metal winner on this test split', '',
              '| Metal | Lowest test RMSE model | RMSE (simulated µg/L) | R² |', '|---|---|---:|---:|']
    for metal, row in winners.iterrows():
        record = metrics[(metrics.metal == metal) & (metrics.model == row['model'])].iloc[0]
        lines.append(f"| {metal} | {row['model']} | {record.RMSE:.3f} | {record.R2:.3f} |")
    lines += ['', 'These per-metal test winners are **not** used to choose the deployed prototype models. '
              'Those were selected by grouped validation within the training sessions; a separate calibration set supplies synthetic prediction intervals.', '',
              '## Interpretation flow for one uploaded image', '',
              '1. Check file, card geometry, lighting and color references. Reject failures before showing concentrations.',
              '2. Detect the fiducials, rectify the card, correct color, and extract pad features.',
              '3. Check whether the feature vector resembles the synthetic training domain. Reject out-of-domain images.',
              '4. Predict seven simulated concentrations. Withhold individual out-of-range predictions rather than clipping them.',
              '5. Show each supported point estimate with its synthetic-session-calibrated interval and any range flag.',
              '6. State that no safety, compliance, or real-world detection decision follows from these numbers.', '',
              '## What can be claimed in a paper', '',
              'The software pipeline recovered simulated concentration signals under the stated renderer and split. '
              'It has **not** established metal selectivity, limit of detection, recovery, reproducibility, or accuracy in real water. '
              'The simulator encodes concentration-dependent colors by assumption, and the training and test images share that simulator and card layout. '
              'Physical seven-metal strips, standardized ESP32-S3 captures, and independent reference-laboratory measurements are required for analytical validation.', '',
              'Source tables: `results/test_metrics.csv`, `results/robustness/reliable_test_metrics.csv`, '
              '`results/reliability_validation.csv`, and `results/calibration_session_scores.csv`.', '']
    return '\n'.join(lines)




## 6. Run or load everything
A fresh folder triggers the full experiment automatically. To force retraining later, set `REBUILD_ALL = True` in section 1. This first trains the baseline and its QC/interval bundle, then independently generates and trains the expanded scenario study. Test/challenge data never tune models.

In [ ]:
if REBUILD_ALL:
    run_all()                     # Original 500-image baseline
    load_reliable.cache_clear()
    train_reliable()              # Training-only model selection and synthetic interval calibration
    main()                        # Expanded 1,800-image scenario study + 120 QC stress cases
else:
    needed = [ROOT/'datasets/labels.csv', ROOT/'results/test_metrics.csv',
              ROOT/'models/reliable/config.json', ROOT/'scenario_study/labels.csv',
              ROOT/'scenario_study/results/metrics_by_scenario_metal.csv']
    missing = [str(p) for p in needed if not p.exists()]
    if missing:
        raise FileNotFoundError('Saved results are incomplete. Set REBUILD_ALL=True. Missing: ' + ', '.join(missing))
    print('Loaded the supplied images, trained models and saved results.')

improved_outputs = [V2_MODELS/'config.json', V2_DIR/'comparison_metrics.csv', V2_DIR/'REPORT.md', V2_DIR/'comparison.png']
if REBUILD_ALL or not all(p.exists() for p in improved_outputs):
    run_improvement()

baseline_labels = pd.read_csv(ROOT/'datasets/labels.csv')
expanded_labels = pd.read_csv(ROOT/'scenario_study/labels.csv')
baseline_metrics = pd.read_csv(ROOT/'results/test_metrics.csv')
scenario_metrics = pd.read_csv(ROOT/'scenario_study/results/metrics_by_scenario_metal.csv')
scenario_qc = pd.read_csv(ROOT/'scenario_study/results/acceptance_by_scenario.csv')
assert len(baseline_labels)==500 and len(expanded_labels)==1800
assert baseline_labels.groupby('session_id').partition.nunique().max()==1
assert expanded_labels.groupby('session_id').partition.nunique().max()==1
assert expanded_labels.groupby('sample_id').partition.nunique().max()==1
print('Original:', len(baseline_labels), 'images;', baseline_labels.sample_id.nunique(), 'profiles')
print('Expanded:', len(expanded_labels), 'images;', expanded_labels.sample_id.nunique(), 'profiles')


## 7. Compare model results without calling them accuracy
MAE/RMSE are simulated concentration errors. R² describes regression fit, not percent correct. NRMSE is scaled by each metal’s assumed full-scale concentration. The shifted-camera challenge is a simulation, not a physical validation.

In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown(build_research_interpretation()))
display(Markdown((ROOT/'scenario_study/REPORT.md').read_text(encoding='utf-8')))
display(scenario_metrics)
display(scenario_qc)
display(Image(filename=str(ROOT/'charts/model_comparison.png')))
display(Image(filename=str(ROOT/'scenario_study/scenario_model_comparison.png')))
display(Image(filename=str(ROOT/'scenario_study/image_acceptance.png')))
display(Markdown((V2_DIR/'REPORT.md').read_text(encoding='utf-8')))
display(pd.read_csv(V2_DIR/'coverage_and_acceptance.csv'))
display(Image(filename=str(V2_DIR/'comparison.png')))


## 8. Predict one included synthetic image
This is a workflow demonstration only. Rejected captures yield no concentrations. A real strip image is not supported without physical calibration and laboratory reference measurements.

In [ ]:
example = baseline_labels[baseline_labels.partition=='test'].iloc[0]
result, corrected = estimate_improved(ROOT/example.image_path)
result['interpretation'] = interpret_prediction(result)
print('Image:', example.image_id)
print('Interpretation:', result['interpretation'])
display(pd.DataFrame(result['predictions']))


## 9. Research boundary

The first 500 and the new 1,800 images are separate synthetic experiments, not 2,300 independent real samples. All concentrations originate from assumed color-generating equations. The expanded study’s intentional corruptions add 120 images but **zero independent profiles**. For real-world claims, manufacture the seven-metal strip, standardize ESP32-S3 capture conditions, collect multiple physical lots and water matrices, pair images with reference-laboratory concentrations, and reserve a fully independent external test campaign.